<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/MCP_Simple_Colab_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP — Beginner Google Colab Lab

**Goal:** Create one MCP tool and call it from a client. **Time:** 20–30 minutes.

Run the three code cells in order. No API key is needed.


### ขั้นตอนที่ 1: ติดตั้งไลบรารี MCP

ในเซลล์นี้ เราจะทำการติดตั้งแพ็กเกจ `mcp` (Model Context Protocol) ซึ่งเป็นไลบรารีหลักที่ใช้สำหรับการพัฒนาทั้งฝั่ง Server และ Client ในที่นี้เราจะระบุเวอร์ชัน `mcp>=1,<2` เพื่อให้เข้ากันได้กับโค้ดตัวอย่าง

In [ ]:
%pip -q install "mcp>=1,<2"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 6.8 MB/s eta 0:00:00


### ขั้นตอนที่ 2: สร้าง Weather Server ของคุณ

ในเซลล์นี้ เราใช้คำสั่งพิเศษของ IPython คือ `%%writefile` เพื่อสร้างไฟล์ชื่อ `weather_server.py`

ภายในไฟล์ประกอบด้วย:
1. การสร้างอินสแตนซ์ของ `FastMCP` ซึ่งช่วยให้การสร้าง Server ง่ายและรวดเร็ว
2. การลงทะเบียนฟังก์ชัน `get_weather` ให้เป็นเครื่องมือ (Tool) ด้วยการใช้เดคอเรเตอร์ `@mcp.tool()` เพื่อให้ตัว Client สามารถเรียกใช้งานได้
3. โค้ดตัวอย่างสำหรับการตอบกลับข้อมูลสภาพอากาศตามชื่อเมืองที่ส่งเข้ามา

In [ ]:
%%writefile weather_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Weather Server")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    # A simple mock response
    city_lower = city.lower().strip()
    if "london" in city_lower:
        return f"The weather in {city} is 15°C and Rainy."
    elif "tokyo" in city_lower:
        return f"The weather in {city} is 22°C and Sunny."
    else:
        return f"The weather in {city} is 20°C and Partly Cloudy."

if __name__ == "__main__":
    mcp.run()

Writing weather_server.py


### ขั้นตอนที่ 3: สร้าง Client เพื่อเชื่อมต่อและใช้งาน Server

ในเซลล์นี้ เราจะทำการเขียนโค้ดฝั่ง Client เพื่อเชื่อมต่อไปยัง Server ที่เราเขียนขึ้นเมื่อสักครู่:
1. กำหนดค่าผ่าน `StdioServerParameters` โดยให้รันไฟล์ `weather_server.py` ด้วย Python ตัวปัจจุบัน (`sys.executable`)
2. เชื่อมต่อผ่าน Standard Input/Output (stdio) และเริ่มระบบด้วย `ClientSession`
3. เรียกดูรายการเครื่องมือทั้งหมดที่มีบนเซิร์ฟเวอร์ด้วย `session.list_tools()`
4. ทดลองเรียกใช้เครื่องมือ `get_weather` โดยส่งพารามิเตอร์เป็นเมือง "Tokyo" และแสดงผลลัพธ์ที่ได้รับกลับมา

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command=sys.executable,
    args=["weather_server.py"]
)

async def main():
    # We pass errlog=sys.__stderr__ to avoid Colab's standard error stream lacking fileno()
    async with stdio_client(server, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Tools:", [tool.name for tool in tools.tools])
            result = await session.call_tool(
                "get_weather", arguments={"city": "Tokyo"}
            )
            print(result.content[0].text)

await main()

Tools: ['get_weather']
The weather in Tokyo is 22°C and Sunny.


## Student exercise

1. Change `City` as your prefer in Cell 3.
2. Change the temperature inside `get_weather()` in Cell 2.
3. Rerun Cell 2, then Cell 3.

**Expected starting output:**

```text
Tools: ['get_weather']
The weather in Tokyo is 22°C and Sunny.
```

**Key concept:** The client starts the server as a subprocess, discovers its tools, and calls one via MCP over stdio. The server usually exits when the client connection closes.
